# 4) Congestion and nodal prices

## Objectives

At the end of this notebook you should be able to:

- compute the **power transfer distribution factors** (PTDFs) of a network and read them
- write the DC-constrained dispatch as an LP with the PTDFs, and derive **nodal prices** (locational marginal prices, LMPs) from its duals: λ plus the PTDF-weighted shadow prices of the congested lines
- reproduce cost and every nodal price in **PyPSA** to 10⁻⁶
- explain why a node can have a price that **no generator offers**, and why the cheapest plant can be **curtailed**
- separate the **cost of congestion** from the **congestion rent** — a social cost and a transfer

Module 03 priced electricity at one node: everybody paid the cost of the last unit dispatched. On a network, the last unit that can **reach** a node may be a different one at every node. This lesson puts lesson 3's DC power flow into Module 03's dispatch LP and lets its duals tell us what electricity is worth where.

In [1]:
import logging
import warnings

import linopy
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pypsa
import xarray as xr

warnings.filterwarnings("ignore")
for name in ("pypsa", "linopy"):
    logging.getLogger(name).setLevel(logging.ERROR)

### The network and its plants

Lesson 3's stylised Croatian 400 kV network, now with generators that compete: wind at Velebit, a hydro plant at Split offered at its water value (Module 03, lesson 7), gas at Rijeka, coal at Osijek and imports at Zagreb. All capacities and costs are **illustrative**.

In [2]:
BUSES = ["Zagreb", "Rijeka", "Split", "Osijek", "Velebit"]
LINES = pd.DataFrame([("Zagreb", "Rijeka", 130), ("Rijeka", "Velebit", 110), ("Velebit", "Split", 160),
                      ("Zagreb", "Split", 320), ("Zagreb", "Osijek", 250), ("Osijek", "Split", 330)],
                     columns=["bus0", "bus1", "km"])
LINES.index = [f"{a}-{b}" for a, b in zip(LINES.bus0, LINES.bus1)]
X_KM = 0.32                                                    # ohm/km, typical 400 kV line
LOAD = pd.Series({"Zagreb": 900, "Rijeka": 300, "Split": 450, "Osijek": 250, "Velebit": 0})[BUSES]
GENS = pd.DataFrame([("wind Velebit", "Velebit", 600, 0.0), ("hydro Split", "Split", 700, 30.0),
                     ("gas Rijeka", "Rijeka", 600, 90.0), ("coal Osijek", "Osijek", 400, 80.0),
                     ("import Zagreb", "Zagreb", 800, 130.0)],
                    columns=["name", "bus", "p_max_MW", "cost_EUR_per_MWh"]).set_index("name")
print(f"load {LOAD.sum():.0f} MW; generation capacity {GENS.p_max_MW.sum():.0f} MW")
GENS

load 1900 MW; generation capacity 3100 MW


                   bus  p_max_MW  cost_EUR_per_MWh
name                                              
wind Velebit   Velebit       600               0.0
hydro Split      Split       700              30.0
gas Rijeka      Rijeka       600              90.0
coal Osijek     Osijek       400              80.0
import Zagreb   Zagreb       800             130.0

### PTDFs: where an injection flows

Lesson 3 solved B θ = P for the angles and turned them into flows. Both steps are linear, so the flows are a fixed matrix times the injections: **flow = PTDF · P**. Entry (ℓ, n) is the share of 1 MW injected at node n — and withdrawn at the reference node, here Zagreb — that flows over line ℓ.

In [3]:
def ptdf(reference="Zagreb"):
    """PTDF matrix (lines x buses) of the DC network, injections withdrawn at the reference bus."""
    idx = {b: i for i, b in enumerate(BUSES)}
    b = 1 / (X_KM * LINES.km.to_numpy() / (400 ** 2 / 100))          # series susceptance, p.u. on 100 MVA
    A = np.zeros((len(LINES), len(BUSES)))                            # incidence: +1 at bus0, -1 at bus1
    A[range(len(LINES)), LINES.bus0.map(idx)] = 1
    A[range(len(LINES)), LINES.bus1.map(idx)] = -1
    B = A.T @ np.diag(b) @ A
    keep = [i for i, name in enumerate(BUSES) if name != reference]
    B_inv = np.zeros_like(B)
    B_inv[np.ix_(keep, keep)] = np.linalg.inv(B[np.ix_(keep, keep)])
    return pd.DataFrame(np.diag(b) @ A @ B_inv, index=LINES.index, columns=BUSES)


PTDF = ptdf()
PTDF.round(3)

                Zagreb  Rijeka  Split  Osijek  Velebit
Zagreb-Rijeka      0.0  -0.786 -0.340  -0.147   -0.604
Rijeka-Velebit     0.0   0.214 -0.340  -0.147   -0.604
Velebit-Split      0.0   0.214 -0.340  -0.147    0.396
Zagreb-Split       0.0  -0.138 -0.425  -0.183   -0.255
Zagreb-Osijek      0.0  -0.076 -0.235  -0.670   -0.141
Osijek-Split       0.0  -0.076 -0.235   0.330   -0.141

The Zagreb column is zero: Zagreb is the reference, so an injection there is also withdrawn there. Read the **Velebit** column: of 1 MW injected at Velebit and taken out at Zagreb, 0.604 MW takes the short way through Rijeka (the −0.604 on Rijeka–Velebit and Zagreb–Rijeka is flow towards Zagreb, against the lines' defined direction) and 0.396 MW the long way through Split, where it splits again — 0.255 MW directly to Zagreb, 0.141 MW through Osijek. Nobody chooses these paths. Power divides over every parallel path in inverse proportion to its reactance; a trade between Velebit and Zagreb loads lines that neither party contracted. These **loop flows** are the reason a network cannot be priced like a set of independent pipes.

### The dispatch LP with line limits

Module 03's dispatch, plus one constraint per line: the flow PTDF · (generation − load) must stay within the line's limit. The LP:

min Σ_g c_g p_g  subject to  Σ_g p_g = Σ_n d_n (dual λ),  −F_ℓ ≤ PTDF_ℓ · (G p − d) ≤ F_ℓ (dual μ_ℓ),  0 ≤ p_g ≤ p̄_g

The **nodal price** at node n is what one more MW of load there would cost. It needs one more MW of generation (λ) and that MW's flow through every congested line (PTDF_ℓn), each valued at the line's shadow price: **LMP_n = λ + Σ_ℓ PTDF_ℓn · μ_ℓ** (with μ_ℓ ≤ 0 for a line at its upper limit, ≥ 0 at its lower limit, in this sign convention).

In [4]:
def dispatch(limit_mw):
    """DC-constrained dispatch with every line limited to limit_mw. Returns cost, dispatch, flows, LMPs, lambda, mu."""
    gi = pd.Index(GENS.index, name="gen")
    G = np.zeros((len(BUSES), len(GENS)))                          # generator-to-bus map
    for k, bus in enumerate(GENS.bus):
        G[BUSES.index(bus), k] = 1
    flow_per_gen = PTDF.to_numpy() @ G                             # line x generator
    flow_of_load = PTDF.to_numpy() @ LOAD.to_numpy()               # flow caused by the loads alone

    m = linopy.Model()
    p = m.add_variables(lower=0, upper=xr.DataArray(GENS.p_max_MW.to_numpy(), coords=[gi]), name="p")
    m.add_constraints(p.sum() == LOAD.sum(), name="balance")
    for l, line in enumerate(LINES.index):
        flow = (p * xr.DataArray(flow_per_gen[l], coords=[gi])).sum()
        m.add_constraints(flow <= limit_mw + flow_of_load[l], name=f"upper {line}")
        m.add_constraints(flow >= -limit_mw + flow_of_load[l], name=f"lower {line}")
    m.add_objective((p * xr.DataArray(GENS.cost_EUR_per_MWh.to_numpy(), coords=[gi])).sum())
    m.solve(solver_name="highs", output_flag=False)

    lam = float(m.constraints["balance"].dual)
    mu = pd.Series({line: float(m.constraints[f"upper {line}"].dual) + float(m.constraints[f"lower {line}"].dual)
                    for line in LINES.index})
    lmp = lam + PTDF.T @ mu
    p_sol = p.solution.to_pandas()
    flows = pd.Series(PTDF.to_numpy() @ (G @ p_sol.to_numpy() - LOAD.to_numpy()), index=LINES.index)
    return {"cost": float(m.objective.value), "dispatch": p_sol, "flows": flows, "lmp": lmp, "lambda": lam, "mu": mu}


def pypsa_dispatch(limit_mw):
    """The same dispatch in PyPSA, from the same tables."""
    n = pypsa.Network()
    n.add("Bus", BUSES, v_nom=400)
    for bus, load in LOAD.items():
        if load:
            n.add("Load", f"{bus} load", bus=bus, p_set=load)
    for name, g in GENS.iterrows():
        n.add("Generator", name, bus=g.bus, p_nom=g.p_max_MW, marginal_cost=g.cost_EUR_per_MWh)
    for name, line in LINES.iterrows():
        n.add("Line", name, bus0=line.bus0, bus1=line.bus1, x=X_KM * line.km, r=0.0, s_nom=limit_mw)
    n.optimize(solver_name="highs", log_to_console=False, progress=False)
    return n


free = dispatch(2000)                         # limits far above any flow
print(f"no congestion: cost {free['cost']:,.0f} EUR/h; one price everywhere: {sorted(set(free['lmp'].round(2)))} EUR/MWh")
free["dispatch"].to_frame("MW").T

no congestion: cost 71,000 EUR/h; one price everywhere: [90.0] EUR/MWh


gen  wind Velebit  hydro Split  gas Rijeka  coal Osijek  import Zagreb
MW          600.0        700.0       200.0        400.0            0.0

Without binding limits the network does not matter for prices: the cheapest plants run — wind, hydro and coal at full output — and the gas plant at Rijeka, partly dispatched at 200 of 600 MW, is the marginal unit and sets **one price, 90 €/MWh, at every node**. This is Module 03's single-node dispatch, with the network only along for the ride. The flows it causes, though, are large: 491 MW on Rijeka–Velebit as Velebit's wind is exported.

### 400 MW per line: congestion

In [5]:
LIMIT = 400
res = dispatch(LIMIT)
n = pypsa_dispatch(LIMIT)
pypsa_lmp = n.buses_t.marginal_price.iloc[0][BUSES]
assert abs(n.objective - res["cost"]) < 1e-6 * res["cost"]
assert np.abs(pypsa_lmp.to_numpy() - res["lmp"].to_numpy()).max() < 1e-6
print(f"own LP = PyPSA: cost {res['cost']:,.1f} EUR/h, every nodal price to 1e-6")
print(f"lambda = {res['lambda']:.2f} EUR/MWh; congested lines (mu, EUR/MWh): "
      f"{res['mu'][res['mu'].abs() > 1e-6].round(2).to_dict()}")
summary = pd.DataFrame({"load_MW": LOAD, "generation_MW": GENS.groupby("bus").apply(lambda t: res["dispatch"][t.index].sum()).reindex(BUSES).fillna(0),
                        "LMP_EUR_per_MWh": res["lmp"]})
summary.round(2)

own LP = PyPSA: cost 81,956.3 EUR/h, every nodal price to 1e-6
lambda = 130.00 EUR/MWh; congested lines (mu, EUR/MWh): {'Zagreb-Rijeka': 86.15, 'Rijeka-Velebit': 129.05}


         load_MW  generation_MW  LMP_EUR_per_MWh
Zagreb       900          15.05           130.00
Rijeka       300         300.00            90.00
Split        450         700.00            56.80
Osijek       250         400.00            98.45
Velebit        0         484.95             0.00

In [6]:
XY = {"Zagreb": (15.98, 45.81), "Rijeka": (14.44, 45.33), "Split": (16.44, 43.51), "Osijek": (18.69, 45.55),
      "Velebit": (15.25, 44.60)}                                  # rough positions, for orientation only
fig, ax = plt.subplots(figsize=(7.5, 5))
for name, line in LINES.iterrows():
    (x0, y0), (x1, y1) = XY[line.bus0], XY[line.bus1]
    loading = abs(res["flows"][name]) / LIMIT
    colour = "tab:red" if loading > 0.999 else "grey"
    ax.plot([x0, x1], [y0, y1], color=colour, lw=1 + 4 * loading, zorder=1)
    if res["flows"][name] < 0:                                   # draw the arrow in the direction power flows
        (x0, y0), (x1, y1) = (x1, y1), (x0, y0)
    ax.annotate("", xy=(x0 + 0.62 * (x1 - x0), y0 + 0.62 * (y1 - y0)), xytext=(x0 + 0.5 * (x1 - x0), y0 + 0.5 * (y1 - y0)),
                arrowprops=dict(arrowstyle="-|>", color="black", lw=1.4, mutation_scale=20), zorder=3)
    ax.text(x0 + 0.38 * (x1 - x0), y0 + 0.38 * (y1 - y0), f"{abs(res['flows'][name]):.0f} MW", fontsize=7, ha="center",
            bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.8), zorder=4)
sc = ax.scatter([XY[b][0] for b in BUSES], [XY[b][1] for b in BUSES], c=res["lmp"][BUSES], cmap="viridis",
                s=420, zorder=2, edgecolor="black", vmin=0, vmax=130)
LABEL_AT = {"Velebit": (-0.3, 0.0, "right")}                     # move labels that would sit on a line
for b in BUSES:
    dx, dy, ha = LABEL_AT.get(b, (0.0, 0.22, "center"))
    ax.text(XY[b][0] + dx, XY[b][1] + dy, f"{b}\n{res['lmp'][b]:.2f} EUR/MWh", ha=ha, va="center" if dy == 0 else "bottom",
            fontsize=8)
fig.colorbar(sc, ax=ax, label="nodal price (EUR/MWh)", shrink=0.8)
ax.set(title="400 MW limits: lines at their limit in red", xlim=(13.8, 19.4), ylim=(43.2, 46.4))
ax.axis("off")
plt.tight_layout()
plt.show()

With every line limited to 400 MW, two lines reach their limit — Zagreb–Rijeka and Rijeka–Velebit, both carrying 400 MW towards Zagreb — and one price becomes five. **Zagreb pays 130 €/MWh**: imports are now marginal there. **Rijeka pays 90 €/MWh**, set by its gas plant, which now runs at 300 MW instead of 200. **Velebit's price falls to 0 €/MWh**: 115 MW of its wind cannot leave and is curtailed, so wind is the marginal unit there. Your own LP and PyPSA agree on the cost and on every price to 10⁻⁶.

The two remaining prices are the instructive ones: **Split 56.80 and Osijek 98.45 €/MWh** — no generator offers either. Hydro at Split and coal at Osijek run at full output, so neither is marginal; their nodes' prices come from the network. One more MW of load at Split can only be served by a combination of marginal units that leaves the two full lines exactly full — some more import at Zagreb and some less curtailment at Velebit — and its price is that combination's cost. The formula does the same arithmetic: 130 + (−0.340) × 86.15 + (−0.340) × 129.05 = 56.80 €/MWh, with 86.15 and 129.05 the shadow prices of the two congested lines. Osijek sits electrically further from the congestion, its PTDFs are smaller, and its price stays closer to Zagreb's.

### Who pays for congestion?

In [7]:
gen_revenue = sum(res["lmp"][GENS.bus[g]] * res["dispatch"][g] for g in GENS.index)
consumer_payment = float((res["lmp"] * LOAD).sum())
rent_from_flows = sum(res["flows"][l] * (res["lmp"][LINES.bus1[l]] - res["lmp"][LINES.bus0[l]]) for l in LINES.index)
rent_from_duals = float((res["mu"].abs() * LIMIT).sum())
print(f"consumers pay        {consumer_payment:12,.0f} EUR/h")
print(f"generators receive   {gen_revenue:12,.0f} EUR/h")
print(f"difference           {consumer_payment - gen_revenue:12,.2f} EUR/h")
print(f"flow x price spread  {rent_from_flows:12,.2f} EUR/h   (summed over lines)")
print(f"|mu| x limit         {rent_from_duals:12,.2f} EUR/h   (summed over congested lines)")
print(f"\ncost of congestion: {res['cost'] - free['cost']:,.0f} EUR/h; wind curtailed: "
      f"{GENS.p_max_MW['wind Velebit'] - res['dispatch']['wind Velebit']:.0f} MW")

consumers pay             194,170 EUR/h
generators receive        108,092 EUR/h
difference              86,077.67 EUR/h
flow x price spread     86,077.67 EUR/h   (summed over lines)
|mu| x limit            86,077.67 EUR/h   (summed over congested lines)

cost of congestion: 10,956 EUR/h; wind curtailed: 115 MW


The accounting closes three ways. Consumers pay 194,170 €/h at their nodal prices, generators receive 108,092 €/h at theirs, and the difference — **86,078 €/h of congestion rent** — equals both the flows times the price differences across the lines and the shadow prices times the line limits. The money is not lost: it is collected by whoever owns the transmission rights — the TSOs, or holders of financial transmission rights — and can pay for new capacity or lower network tariffs.

The **cost of congestion** is something else: what society loses because the cheap power cannot get through. It is 10,956 €/h — 115 MW of free wind replaced by 100 MW of gas at 90 €/MWh and 15 MW of imports at 130 €/MWh. The rent is almost eight times larger than the cost, because the rent is a **transfer** that shifts money between consumers, generators and the network owner, while the cost is the resources actually wasted. Confusing the two is a classic mistake in public debate about grid expansion: a line's value is the congestion cost it removes, not the rent it ends.

## Check your understanding

**PTDFs and prices**

1. Read the PTDF column for Velebit: of 1 MW injected there and withdrawn at Zagreb, how much flows over Zagreb–Rijeka, and how much takes the long way round through Split?
2. Split and Osijek get prices that no generator offers. Using LMP = λ + Σ PTDF · μ, explain where Split's 56.80 EUR/MWh comes from.

**Money**

1. The congestion rent is eight times the cost of congestion. Why is one a cost to society and the other not?
2. Who receives the congestion rent in a real market, and what could it be used for?